# Political Fairness Audit

Extension of the *Moral Frame Preserving News Summarization* paper (Bondielli et al., COLM 2025) into a political-fairness analysis.

**Goal:** check whether the paper's prompting methods (Plain / Direct / CoT / Oracle / Class) preserve moral framing *equally* across left-, center-, and right-leaning news articles, and whether the gap depends on the prompting strategy.

**Data:** the AllSides subset of EMONA, where every article filename encodes its political leaning (`allsides_<topic>_<l|c|r>_<n>`). The paper ran three models (Llama-3-70B, CommandR-Plus, DeepSeek-R1-Distill-Qwen-32B) on **180 AllSides articles each** (60 left / 60 center / 60 right, across 12 topics).

This notebook focuses on Llama-3-70B for the deep dive; cross-model comparison is a follow-up.

All data is loaded into in-memory DataFrames -- no CSV side-effects.

## 1. Load the data

`data_loader.py` lives in the repo root. It pulls the three precomputed metric pickles in `results/automated_evaluation/`, normalizes the `QaFactEval`/`QAFactEval` key inconsistency, drops the `mean`/`std` summary rows, deduplicates rows that appear in more than one pickle (preferring the larger-coverage source), and parses topic/leaning out of the AllSides article IDs.

In [2]:
import sys
from pathlib import Path

# Make sure we can import data_loader.py from the repo root.
REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import pandas as pd

from data_loader import (
    build_full_long,
    build_wide_by_method,
    allsides_only,
    summary_counts,
    PROMPTING_METHODS,
    PAPER_NAME_MAP,
)

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)

In [3]:
# Build the master long-format and wide-format DataFrames once.
long_df = build_full_long()
wide_df = build_wide_by_method(long_df)

print(f"long_df: {len(long_df):,} rows  -- one row per (model, article, prompting_method, metric)")
print(f"wide_df: {len(wide_df):,} rows  -- one row per (model, article, metric); prompting methods are columns")

long_df: 38,400 rows  -- one row per (model, article, prompting_method, metric)
wide_df: 7,165 rows  -- one row per (model, article, metric); prompting methods are columns


## 2. Sanity check: counts per leaning

We expect **180 AllSides articles per model**, balanced 60 left / 60 center / 60 right.

In [4]:
summary_counts(long_df)

leaning,model,center,left,right
0,DeepSeek-R1-Distill-Qwen-32B,60,60,60
1,Meta-Llama-3-70B-Instruct,60,60,60
2,c4ai-command-r-plus-4bit,60,60,60


### Topic x leaning breakdown

EMONA-AllSides is a fully balanced factorial: 12 topics x 3 leanings x 5 replicate articles.

In [4]:
topic_breakdown = (
    allsides_only(long_df)
    .query("model == 'Meta-Llama-3-70B-Instruct' and metric == 'moral_count' and prompting_method == 'vanilla'")
    .groupby(["topic", "leaning"]).size()
    .unstack("leaning", fill_value=0)
)
topic_breakdown

leaning,center,left,right
topic,,,
abortion,5,5,5
coronavirus,5,5,5
elections,5,5,5
gun_control_and_gun_rights,5,5,5
immigration,5,5,5
lgbt_rights,5,5,5
politics,5,5,5
race_and_racism,5,5,5
us_house,5,5,5


## 3. First peek: mean moral_count by leaning x method (Llama)

Raw moral-word retention by political leaning. This is **not** the retention rate yet -- it's the absolute count of moral words preserved in the summary. Left-leaning articles may simply contain more moral language to begin with, which would show up as a higher absolute count even under fair preservation.

We'll convert to a proper retention rate (`moral_count / original`) in the next notebook.

In [7]:
llama_moral_count = (
    allsides_only(long_df)
    .query("model == 'Meta-Llama-3-70B-Instruct' and metric == 'moral_count'")
    .query("prompting_method in @PROMPTING_METHODS")
)

peek = (
    llama_moral_count
    .groupby(["leaning", "prompting_method"])["value"]
    .mean()
    .unstack("prompting_method")
    .round(2)
    # Reorder columns into the paper's naming order: Plain, Direct, CoT, Oracle, Class
    [PROMPTING_METHODS]
    .rename(columns=PAPER_NAME_MAP)
)
peek

prompting_method,Plain,Direct,CoT,Oracle,Class
leaning,,,,,
center,6.48,6.99,7.92,10.86,10.04
left,7.85,8.40,9.19,12.86,11.13
right,7.54,8.27,9.07,12.14,8.67


Also handy: the per-article original (source) moral-word count, by leaning. This tells us whether left/center/right articles differ in baseline moral density.

In [8]:
original_moral_density = (
    allsides_only(long_df)
    .query("model == 'Meta-Llama-3-70B-Instruct' and metric == 'moral_count' and prompting_method == 'original'")
    .groupby("leaning")["value"]
    .agg(["mean", "std", "count"])
    .round(2)
)
original_moral_density

,mean,std,count
leaning,,,
center,21.53,10.63,60
left,27.82,12.99,60
right,23.72,12.80,60
